<a href="https://colab.research.google.com/github/JoelForson/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Lab%206/The_Architecture_of_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Data Ingestion (The Population)
df = sns.load_dataset('titanic')
print(f"Total Population: {len(df)}")
print(f"Population Survival Rate: {df['survived'].mean():.4f}")

# 2. Manual Shuffle (Simulation of Sampling)
# We set a seed to ensure reproducibility for the lesson,
# but in production, this variance happens naturally.
np.random.seed(2026)
indices = np.random.permutation(df.index)

Total Population: 891
Population Survival Rate: 0.3838


In [5]:
df.describe().round(2)

,survived,pclass,age,sibsp,parch,fare
count,891.00,891.00,714.00,891.00,891.00,891.00
mean,0.38,2.31,29.70,0.52,0.38,32.20
std,0.49,0.84,14.53,1.10,0.81,49.69
min,0.00,1.00,0.42,0.00,0.00,0.00
25%,0.00,2.00,20.12,0.00,0.00,7.91
50%,0.00,3.00,28.00,0.00,0.00,14.45
75%,1.00,3.00,38.00,1.00,0.00,31.00
max,1.00,3.00,80.00,8.00,6.00,512.33


# **Step 2: The Split and Bias Check**
We will split the data 80/20. Then we will calculate the survival rate in both groups, should be identical

In [8]:
# 3. Cut the deck (80/20 Split)
split_point = int(len(df.index) * 0.8)

# Slicing the shuffled indices
train_idx = indices[:split_point]
test_idx = indices[split_point:]

# Creating the subsets
train_set = df.loc[train_idx]
test_set = df.loc[test_idx]

# 4. Bias Check (The Delta)
train_surv = train_set['survived'].mean()
test_surv = test_set['survived'].mean()
delta = abs(train_surv - test_surv)

print(f"Train Survival Rate: {train_surv:.4f}")
print(f"Test Survival Rate:  {test_surv:.4f}")
print(f"Sampling Bias (Delta): {delta:.4f}")

Train Survival Rate: 0.3736
Test Survival Rate:  0.4246
Sampling Bias (Delta): 0.0510


# **Step 3: Fixing Covariate Shift**
We suspect that "Class" (pclass) is a major driver of survival. A random split might accidentally put all First Class passengers in the Training set. We must force the distribution to be identical.

In [9]:
df.corr(numeric_only = True)


,survived,pclass,age,sibsp,parch,fare,adult_male,alone
survived,1.000000,-0.338481,-0.077221,-0.035322,0.081629,0.257307,-0.557080,-0.203367
pclass,-0.338481,1.000000,-0.369226,0.083081,0.018443,-0.549500,0.094035,0.135207
age,-0.077221,-0.369226,1.000000,-0.308247,-0.189119,0.096067,0.280328,0.198270
sibsp,-0.035322,0.083081,-0.308247,1.000000,0.414838,0.159651,-0.253586,-0.584471
parch,0.081629,0.018443,-0.189119,0.414838,1.000000,0.216225,-0.349943,-0.583398
fare,0.257307,-0.549500,0.096067,0.159651,0.216225,1.000000,-0.182024,-0.271832
adult_male,-0.557080,0.094035,0.280328,-0.253586,-0.349943,-0.182024,1.000000,0.404744
alone,-0.203367,0.135207,0.198270,-0.584471,-0.583398,-0.271832,0.404744,1.000000


In [11]:
from sklearn.model_selection import train_test_split

# Stratify by 'pclass' ensures the distribution of classes is identical
X_train, X_test = train_test_split(df, test_size=0.2)

print("\n--- Stratified Split ---")
print("Train Class Dist:\n", X_train['pclass'].value_counts(normalize=True))
print("Test Class Dist:\n", X_test['pclass'].value_counts(normalize=True))


--- Stratified Split ---
Train Class Dist:
 pclass
3    0.544944
1    0.247191
2    0.207865
Name: proportion, dtype: float64
Test Class Dist:
 pclass
3    0.575419
1    0.223464
2    0.201117
Name: proportion, dtype: float64


In [27]:
population_probs = df['pclass'].value_counts(normalize=True).sort_index()

sample_size = 100

bad_sample = df.sample(n=sample_size, random_state = 2026)

observed = bad_sample['pclass'].value_counts().sort_index().values
expected = (population_probs * sample_size).values
print(f"Observed Counts(Sample): {observed}")
print(f"Expected Counts(Ideal): {expected}")

from scipy.stats import chisquare

chi2_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print("--"*20)
print(f"\nChi-Square Statistic: {chi2_stat:.4f}")

if p_value < 0.01:
    print("CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected")
else:
    print("PASS: Variance is within natural limits.")

Observed Counts(Sample): [21 22 57]
Expected Counts(Ideal): [24.24242424 20.65095398 55.10662177]
----------------------------------------

Chi-Square Statistic: 0.5869
PASS: Variance is within natural limits.


# **Step 4: The SRM Diagnostic (Forensics)**

In "Tech Economics," data streams are often broken by engineering bugs. This leads to Sample Ratio Mismatch (SRM).

Action: Use the prompt below to generate a forensic tool that detects if an A/B test is biased.

In [15]:
import scipy.stats as stats

# A/B Test Sample Ratio Mismatch (SRM) Check
# ==========================================

# Define observed and expected frequencies
observed = [450, 550]  # [Control, Treatment]
expected = [500, 500]  # Expected 50/50 split

# Perform Chi-Square Goodness-of-Fit Test
chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)

# Display results
print("A/B Test Forensic Check: Sample Ratio Mismatch (SRM)")
print("=" * 60)
print(f"Observed:  Control = {observed[0]}, Treatment = {observed[1]}")
print(f"Expected:  Control = {expected[0]}, Treatment = {expected[1]}")
print(f"\nChi-Square Statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print("=" * 60)

# Statistical conclusion
if p_value < 0.01:
    print("\n⚠️  CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected.")
    print("    Action Required: Check Load Balancer.")
else:
    print("\n✓  Variance is within natural limits.")

A/B Test Forensic Check: Sample Ratio Mismatch (SRM)
Observed:  Control = 450, Treatment = 550
Expected:  Control = 500, Treatment = 500

Chi-Square Statistic: 10.0000
P-value: 0.001565

⚠️  CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected.
    Action Required: Check Load Balancer.
